# Genie space CI/CD with Declarative Automation Bundles
### Version history, rollback and promotion dev → qa → prod, with DAB as the only deployment mechanism

**The approved architecture:** the FreshCart Genie space (a Genie Agent in the UI) and everything that runs in
Databricks for it are defined in **one Declarative Automation Bundle** (`genie_bundle/`). Every environment is
deployed with **`databricks bundle deploy -t <target>`**. Nothing else writes to a Genie space.

| Requirement | How it is met with the bundle |
|---|---|
| Every change made in the dev UI has a version | `bundle generate genie-space` exports dev hourly into the bundle; each change is a git commit tagged `genie-dev-snapshot-<time>` |
| Roll back after a mistake | redeploy an earlier version: `git checkout <tag> -- <space file>` + `bundle deploy` (a workflow does it) |
| Promote dev → qa → prod | one bundle, four **targets**; the same commit is deployed to each; `${var.catalog}` points each at its own data |
| Quality gates | `bundle validate` + SQL tests on every PR; Genie **benchmarks** in qa and a smoke test in prod, run as a **bundle job** (`bundle run genie_quality_gate`) |
| Protection | approval before prod; `bundle plan` before every deploy: refuse to overwrite UI edits in qa/prod, refuse to delete/recreate the space, back up the live space |
| Audit trail | git history and tags, GitHub deployment history, the bundle's deployment state, the backup artifacts |

Part 6 runs the **real Databricks CLI** (`databricks bundle validate / deploy / plan / summary / generate`) against
three local stand-in workspaces (dev, qa, prod), so you can watch the whole lifecycle without a Databricks account.
Part 7 shows how to connect the same bundle and workflows to your real workspaces.

## Contents
| Part | |
|---|---|
| 1 | Why DAB, and the three facts that shape the design |
| 2 | The architecture in one picture |
| 3 | The bundle, file by file |
| 4 | Targets: one definition, four environments |
| 5 | The helper scripts (and why they never deploy) |
| 6 | **Hands-on**: the full lifecycle with the real Databricks CLI |
| 7 | Going live: one-time setup on Databricks and GitHub |
| 8 | The CI/CD workflows, job by job |
| 9 | Runbook |
| 10 | Summary and glossary |

**To run:** install the Databricks CLI **1.14 or newer**
(`curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh`), then
`pip install -r notebooks/requirements.txt -r genie_bundle/requirements-dev.txt` and *Run All* (about 1 minute).

---
# 1 · Why DAB, and three facts that shape the design

**Declarative Automation Bundles** (formerly Databricks Asset Bundles) describe Databricks resources in YAML and
deploy them per **target** (environment) with one command. Since CLI 1.3 bundles have a native **`genie_spaces`**
resource (direct deployment engine, the default since CLI 1.14). Using DAB for the Genie space means it is
deployed, promoted and permissioned exactly like jobs, pipelines and dashboards: one tool, one review process.

We checked the CLI's behaviour (v1.18.0) before designing; three facts matter:

| # | Fact (verified with the real CLI) | Consequence for the design |
|---|---|---|
| 1 | A space defined with `file_path: x.geniespace.json` is inlined **as-is**: `${var.catalog}` inside the file is **not** resolved (and a literal `${...}` breaks `bundle plan`) | The space content is kept as a **bundle variable** in YAML, where `${var.catalog}` **is** resolved per target |
| 2 | `bundle generate genie-space --existing-id <id>` exports a space as JSON with the environment's real catalog names | A small script converts the export back to `${var.catalog}` form (`sync_from_workspace.py`) |
| 3 | A non-interactive `bundle deploy` overwrites edits made in the UI since the last deploy **without asking**. `bundle plan -o json` reveals them: the space's `etag` moved (`changes.etag.old != changes.etag.remote`) | CI runs `bundle plan` before every qa/prod deploy and stops on such drift (`check_drift.py`), keeping a backup of the live space from the plan |

---
# 2 · The architecture in one picture

```
 DEVELOPERS                          GIT (GitHub)                              RELEASE (genie-release, all `databricks bundle ...`)
 ──────────                          ────────────                              ───────────────────────────────────────────────────
 edit the DEV space in the Genie UI
        │
        │ hourly genie-dev-sync:
        │   bundle summary + bundle generate genie-space
        ├────────────────────────────► genie_bundle/resources/freshcart_assistant.space.yml
        │                              commit + tag genie-dev-snapshot-<time>  (every UI change = a version)
        │                                     │
        │                              pull request ── ci + genie-ci: tests · validate_space --sql · bundle validate
        │                                     │        · readable change list · reviewer approval
        │                                     ▼
        │                              merge to main ───────────────────────►  dev:  snapshot first, then bundle deploy -t dev
        │                                                                       qa:   bundle plan → drift gate → backup → bundle deploy -t qa
        │                                                                             → bundle run genie_quality_gate (benchmarks) → smoke
        │                                                                       [ approval (prod environment reviewers) ]
        │                                                                       prod: bundle plan → drift gate → backup → bundle deploy -t prod
        │                                                                             → smoke (fails? redeploy previous release) → tag genie-prod-<version>
        ▼
 genie-rollback (manual): checkout <tag> -- space file (or the whole bundle) → bundle plan → backup → bundle deploy -t <env> → smoke
```

### Where the version history lives
| Record | What | Where |
|---|---|---|
| `genie-dev-snapshot-*` tags | every state of the dev space captured by dev-sync (UI edits) | git |
| commits on `main` | every reviewed change | git |
| `genie-prod-*` tags | every prod release | git |
| pre-deploy backups | the live space (and the full plan) just before each qa/prod deploy or rollback | GitHub workflow artifacts (400 days for prod) |
| deployments | who deployed which commit to which environment, approvals | GitHub environments; the bundle's deployment state in the workspace |

---
# 3 · The bundle, file by file

```
genie_bundle/
├── databricks.yml                              bundle, variables, targets sandbox / dev / qa / prod
├── resources/
│   ├── freshcart_assistant.genie_space.yml     the Genie space resource: title, warehouse, permissions
│   ├── freshcart_assistant.space.yml           the space's content, as the variable freshcart_assistant_space
│   └── genie_quality_gate.job.yml              benchmark gate + smoke test job (nightly monitor in prod)
├── src/genie_quality_gate.py                   the job's task (runs in Databricks)
├── scripts/                                    local / CI helpers (never deploy; see part 5)
│   ├── validate_space.py   sync_from_workspace.py   check_drift.py   space_tools.py
└── requirements-dev.txt
```

In [1]:
import os, sys, json, shutil, subprocess, tempfile, re, copy
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "genie_bundle").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT / "genie_bundle" / "scripts"), str(ROOT / "tests"), str(ROOT)]

import pandas as pd
import space_tools as T

CLI = os.environ.get("DATABRICKS_CLI") or shutil.which("databricks")
version = subprocess.run([CLI, "--version"], capture_output=True, text=True).stdout.strip() if CLI else None
assert version and tuple(map(int, re.search(r"v(\d+)\.(\d+)", version).groups())) >= (1, 14), \
    "Install the Databricks CLI 1.14+: curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh"
print(version)

Databricks CLI v1.18.0


### `databricks.yml`: the bundle, its variables and targets

In [2]:
print((ROOT / "genie_bundle/databricks.yml").read_text())

# FreshCart Sales & Stock Assistant: Genie space delivered with a Declarative Automation Bundle
# ------------------------------------------------------------------------------------------------
# Everything that runs in Databricks for the Genie space is defined in this bundle and deployed with
#     databricks bundle deploy -t <sandbox | dev | qa | prod>
#
#   resources/freshcart_assistant.genie_space.yml   the Genie space: title, warehouse, permissions
#   resources/freshcart_assistant.space.yml         the space's content (instructions, examples,
#                                                   benchmarks...) as the variable freshcart_assistant_space
#   resources/genie_quality_gate.job.yml            benchmark gate + smoke test (and nightly monitor in prod)
#
# The space content refers to tables as ${var.catalog}.gold.dim_store, so one definition serves every
# target: the bundle resolves the catalog for each target at deploy time.
#
# Requires Databricks CLI 1.14+ (Genie spaces

### The Genie space resource
The resource holds only the settings people edit by hand. Its content comes from the variable
`freshcart_assistant_space`, defined in the next file. (Bundles don't allow one resource to be split across two
files, but a resource may take a whole value from a complex variable, and that variable can live in its own file.)

In [3]:
print((ROOT / "genie_bundle/resources/freshcart_assistant.genie_space.yml").read_text())

# The Genie space (a Genie Agent in the UI). Edit these settings by hand.
# Its content lives in freshcart_assistant.space.yml (variable freshcart_assistant_space).
resources:
  genie_spaces:
    freshcart_assistant:
      title: FreshCart Sales & Stock Assistant${var.title_suffix}
      description: >-
        Ask about sales, margin, baskets, promotions, loyalty and stock across FreshCart stores
        and online. Figures are USD, exclude tax and use the FreshCart fiscal calendar.
        You only see regions you have access to; if a question returns no rows, you may not
        have access to that region. Not covered: labour, supplier deliveries, individual customers.
      warehouse_id: ${var.warehouse_id}
      serialized_space: ${var.freshcart_assistant_space}
      permissions:
        - level: CAN_RUN
          group_name: ${var.business_users_group}



### The space's content
This is Databricks' serialized-space format (what `GET /api/2.0/genie/spaces/{id}?include_serialized_space=true`
returns), written as YAML: one line per SQL line, sorted, so pull-request diffs are short and exact. Every table
is written `${var.catalog}.<schema>.<object>`.

In [4]:
space = T.load_space()
print("content hash:", T.content_hash(space))
display(pd.DataFrame(T.summary(space).items(), columns=["section", "items"]))
text = (ROOT / "genie_bundle/resources/freshcart_assistant.space.yml").read_text().splitlines()
start = next(i for i, l in enumerate(text) if "example_question_sqls" in l)
print("\n".join(text[start:start + 16]))

content hash: 2fc87b54a109


,section,items
0,table,2
1,metric view,2
2,sample question,4
3,text instruction,1
4,example SQL,6
5,trusted function,1
6,SQL filter,2
7,SQL expression,1
8,benchmark,17


        example_question_sqls:
        - id: 08131930aafcd3a8773021c860fd5d37
          question:
          - What were net sales and margin by region last week?
          sql:
          - "SELECT region,\n"
          - "       MEASURE(net_sales)        AS net_sales,\n"
          - "       MEASURE(gross_margin)     AS gross_margin,\n"
          - "       MEASURE(gross_margin_pct) AS gross_margin_pct\n"
          - "FROM   ${var.catalog}.semantic.sales_metrics\n"
          - "WHERE  is_last_completed_fiscal_week\n"
          - "GROUP BY ALL\n"
          - ORDER BY net_sales DESC
        - id: 0f51611b4aa70d04c04fb8a612a99516
          question:
          - Which categories rely most on promotions this quarter?


### The quality-gate job
Deployed with the space, run with `databricks bundle run genie_quality_gate -t <target>`. It gets the space id from
the bundle itself: `${resources.genie_spaces.freshcart_assistant.id}`. In prod, `databricks.yml` adds a nightly
schedule, so the benchmarks also run every morning on live data and alert on regressions.

In [5]:
print((ROOT / "genie_bundle/resources/genie_quality_gate.job.yml").read_text())

# Quality gates for the Genie space, deployed with the bundle and run with `databricks bundle run`:
#   databricks bundle run genie_quality_gate -t qa                        benchmark gate (fails the run if it fails)
#   databricks bundle run genie_quality_gate -t prod --params mode=smoke  one question must return SQL
# In prod the job is also scheduled nightly (see databricks.yml) to catch regressions on live data.
resources:
  jobs:
    genie_quality_gate:
      name: FreshCart Genie quality gate${var.title_suffix}
      description: Runs the Genie space's benchmarks (mode=gate) or one smoke-test question (mode=smoke).
      max_concurrent_runs: 1
      parameters:
        - name: space_id
          default: ${resources.genie_spaces.freshcart_assistant.id}
        - name: mode
          default: gate
        - name: min_accuracy
          default: ${var.gate_min_accuracy}
        - name: max_bad
          default: ${var.gate_max_bad}
        - name: min_graded
          default: ${va

<details>
<summary><b>📄 The job's task: <code>genie_bundle/src/genie_quality_gate.py</code></b> (click to open)</summary>

```python
"""Quality gate for the FreshCart Genie space. Runs as the task of the bundle job `genie_quality_gate`.

    mode=gate   run every benchmark of the space (Genie benchmark evaluation) and fail unless
                  - no benchmark is answered incorrectly (BAD <= max_bad),
                  - at least min_graded benchmarks could be graded automatically, and
                  - accuracy on the graded ones is >= min_accuracy
    mode=smoke  ask one question; fail unless Genie answers with SQL

A failed check raises SystemExit(1), so the job run fails and `databricks bundle run` exits non-zero,
which stops the release pipeline. Authentication is the job's identity (run_as) inside Databricks;
run locally, it uses the standard Databricks environment variables or profile.
"""
from __future__ import annotations

import argparse
import sys
import time
from datetime import timedelta

from databricks.sdk import WorkspaceClient


def _value(x):
    return getattr(x, "value", x)


def run_benchmarks(w: WorkspaceClient, space_id: str, timeout_s: int = 1800, poll_s: int = 10) -> dict:
    run = w.genie.genie_create_eval_run(space_id)
    deadline = time.time() + timeout_s
    while _value(run.eval_run_status) in (None, "NOT_STARTED", "RUNNING"):
        if time.time() > deadline:
            raise TimeoutError(f"benchmark run {run.eval_run_id} did not finish within {timeout_s}s")
        time.sleep(poll_s)
        run = w.genie.genie_get_eval_run(space_id, run.eval_run_id)
    results, token = [], None
    while True:
        page = w.genie.genie_list_eval_results(space_id, run.eval_run_id, page_size=100, page_token=token)
        for r in page.eval_results or []:
            d = w.genie.genie_get_eval_result_details(space_id, run.eval_run_id, r.result_id)
            reasons = "; ".join(str(_value(x)) for x in (d.assessment_reasons or []))
            results.append({"question": r.question, "assessment": _value(d.assessment) or "NEEDS_REVIEW", "reason": reasons})
        token = page.next_page_token
        if not token:
            break
    return {"eval_run_id": run.eval_run_id, "status": _value(run.eval_run_status),
            "num_questions": run.num_questions or len(results), "num_correct": run.num_correct or 0,
            "num_needs_review": run.num_needs_review or 0,
            "num_bad": sum(1 for r in results if r["assessment"] == "BAD"), "results": results}


def evaluate(res: dict, *, min_accuracy: float, max_bad: int, min_graded: int) -> dict:
    graded = res["num_questions"] - res["num_needs_review"]
    accuracy = res["num_correct"] / graded if graded else 0.0
    reasons = []
    if res["status"] != "DONE":
        reasons.append(f"the benchmark run ended with status {res['status']}")
    if res["num_bad"] > max_bad:
        reasons.append(f"{res['num_bad']} benchmark(s) answered incorrectly (allowed {max_bad})")
    if graded < min_graded:
        reasons.append(f"only {graded} benchmark(s) could be graded (need {min_graded})")
    if accuracy < min_accuracy:
        reasons.append(f"accuracy {accuracy:.0%} is below {min_accuracy:.0%}")
    return {**res, "graded": graded, "accuracy": accuracy, "passed": not reasons, "reasons": reasons}


def gate(w, space_id, *, min_accuracy, max_bad, min_graded) -> dict:
    res = evaluate(run_benchmarks(w, space_id), min_accuracy=min_accuracy, max_bad=max_bad, min_graded=min_graded)
    print(f"benchmark run {res['eval_run_id']}: {res['num_correct']}/{res['graded']} graded correct, "
          f"{res['num_bad']} bad, {res['num_needs_review']} need review")
    for r in res["results"]:
        if r["assessment"] == "BAD":
            print(f"  BAD  {r['question']}\n       {r['reason']}")
    print("GATE PASSED" if res["passed"] else "GATE FAILED: " + "; ".join(res["reasons"]))
    return res


def smoke(w, space_id, question, timeout_s: int = 600) -> dict:
    msg = w.genie.start_conversation_and_wait(space_id, question, timeout=timedelta(seconds=timeout_s))
    sql = next((a.query.query for a in (msg.attachments or []) if a.query and a.query.query), None)
    status = _value(msg.status)
    passed = status == "COMPLETED" and bool(sql)
    print(f"smoke test: {question!r} -> {status}{' with SQL' if sql else ' without SQL'}")
    if sql:
        print("  " + sql.replace("\n", "\n  "))
    print("SMOKE TEST PASSED" if passed else "SMOKE TEST FAILED")
    return {"status": status, "sql": sql, "passed": passed}


def main(argv=None, client=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--space-id", required=True)
    ap.add_argument("--mode", choices=["gate", "smoke"], default="gate")
    ap.add_argument("--min-accuracy", type=float, default=0.95)
    ap.add_argument("--max-bad", type=int, default=0)
    ap.add_argument("--min-graded", type=int, default=1)
    ap.add_argument("--smoke-question", default="What were net sales and margin by region last week?")
    args = ap.parse_args(argv)
    w = client or WorkspaceClient()
    if args.mode == "gate":
        ok = gate(w, args.space_id, min_accuracy=args.min_accuracy, max_bad=args.max_bad, min_graded=args.min_graded)["passed"]
    else:
        ok = smoke(w, args.space_id, args.smoke_question)["passed"]
    return 0 if ok else 1


if __name__ == "__main__":
    code = main()
    if code:
        sys.exit(code)        # a non-zero exit fails the job task, and with it `databricks bundle run`
```
</details>

---
# 4 · Targets: one definition, four environments

| Target | Mode | Who deploys | Where | Catalog | Purpose |
|---|---|---|---|---|---|
| `sandbox` | development | each developer, from a laptop | their user folder; title `[dev <name>] ...` | `freshcart_dev` | personal experiments |
| `dev` | production | CI (dev service principal) | the principal's folder; title `... [DEV]` | `freshcart_dev` | the shared space the team edits in the UI |
| `qa` | production | CI (qa service principal) | `... [QA]` | `freshcart_qa` | benchmark gate before prod |
| `prod` | production | CI (prod service principal), after approval | `FreshCart Sales & Stock Assistant` | `freshcart` | business users (CAN_RUN) |

`bundle validate` shows what each target will deploy. This runs the real CLI against a local stand-in workspace
(started below); look at the catalog in the table names:

In [6]:
from fake_workspace import FakeWorkspace, FreshCartEvaluator

def clean(out: str) -> str:
    # drop CLI hints that are irrelevant here (skills, OIDC discovery on a local host)
    return "\n".join(l for l in out.splitlines() if "skills are not installed" not in l and "Host metadata" not in l).strip()

probe = FakeWorkspace.start()
env = {**os.environ, "DATABRICKS_HOST": probe.host, "DATABRICKS_TOKEN": "local"}
rows = []
for target in ("sandbox", "dev", "qa", "prod"):
    out = subprocess.run([CLI, "bundle", "validate", "-t", target, "-o", "json"], cwd=ROOT / "genie_bundle",
                         env=env, capture_output=True, text=True).stdout
    cfg = json.loads(out)
    gs = cfg["resources"]["genie_spaces"]["freshcart_assistant"]
    tables = [t["identifier"] for t in json.loads(gs["serialized_space"])["data_sources"]["tables"]]
    job = cfg["resources"]["jobs"]["genie_quality_gate"]
    rows.append({"target": target, "space title": gs["title"], "tables": ", ".join(tables),
                 "nightly gate": bool(job.get("schedule")), "root path": cfg["workspace"]["root_path"]})
probe.stop()
pd.DataFrame(rows)

,target,space title,tables,nightly gate,root path
0,sandbox,[dev dana] FreshCart Sales & Stock Assistant,"freshcart_dev.gold.dim_product, freshcart_dev....",False,/Workspace/Users/dana@freshcart.example/.bundl...
1,dev,FreshCart Sales & Stock Assistant [DEV],"freshcart_dev.gold.dim_product, freshcart_dev....",False,/Workspace/Users/dana@freshcart.example/.bundl...
2,qa,FreshCart Sales & Stock Assistant [QA],"freshcart_qa.gold.dim_product, freshcart_qa.go...",False,/Workspace/Users/dana@freshcart.example/.bundl...
3,prod,FreshCart Sales & Stock Assistant,"freshcart.gold.dim_product, freshcart.gold.dim...",True,/Workspace/Users/dana@freshcart.example/.bundl...


---
# 5 · The helper scripts, and why they never deploy

Deployment is **only** ever `databricks bundle deploy`. The scripts fill the three gaps from part 1 and run in CI or
on a laptop; they talk to Databricks only through the `databricks` CLI.

| Script | Used by | What it does |
|---|---|---|
| `validate_space.py [--sql] [--diff-against REF] [--fix]` | genie-ci on every PR | structure, unique ids, `${var.catalog}` everywhere (no hard-coded `freshcart_qa.`), canonical form; `--sql` runs every example and benchmark SQL on the local FreshCart warehouse; `--diff-against` prints the readable change list |
| `sync_from_workspace.py -t <target>` | genie-dev-sync (hourly), developers | `bundle summary` → space id; `bundle generate genie-space --existing-id` → JSON; catalog → `${var.catalog}`; writes the space file |
| `check_drift.py plan.json -t <target>` | genie-release, genie-rollback | reads `bundle plan -o json`: backs up the live space, exits 2 on UI edits since the last deploy, exits 3 if the space would be deleted or recreated (users would lose their conversations) |
| `src/genie_quality_gate.py` | the bundle job | Genie benchmark evaluation with pass rules, or a smoke-test question |

In [7]:
print(subprocess.run([sys.executable, "genie_bundle/scripts/validate_space.py", "--sql"], capture_output=True, text=True).stdout)

space content 2fc87b54a109: {'table': 2, 'metric view': 2, 'sample question': 4, 'text instruction': 1, 'example SQL': 6, 'trusted function': 1, 'SQL filter': 2, 'SQL expression': 1, 'benchmark': 17}
OK



---
# 6 · Hands-on: the full lifecycle with the real Databricks CLI

We start **three stand-in workspaces** (dev, qa, prod): small local servers implementing the Databricks REST
endpoints the bundle uses (workspace files, Genie spaces, jobs, permissions) plus Genie benchmark runs. The
benchmark grader is not an AI: it answers a question with the space's trusted example SQL for that exact question
and compares the result with the benchmark's SQL on the local FreshCart data (otherwise `NEEDS_REVIEW`).

Everything else is real: the **Databricks CLI** runs `bundle validate / deploy / plan / summary / generate`, and
the gate script uses the **Databricks SDK**. We work in a scratch git repository holding a copy of the bundle.

### 6.1 · Set up the lab

In [8]:
LAB = Path(tempfile.mkdtemp(prefix="genie_dab_lab_"))
REPO = LAB / "repo"
BUNDLE = REPO / "genie_bundle"
shutil.copytree(ROOT / "genie_bundle", BUNDLE, ignore=shutil.ignore_patterns(".databricks", "__pycache__"))
SPACE_FILE = BUNDLE / "resources" / "freshcart_assistant.space.yml"
shutil.copy(ROOT / ".gitignore", REPO / ".gitignore")          # ignores the CLI's local .databricks/ cache

evaluator = FreshCartEvaluator()
WS = {t: FakeWorkspace.start(evaluator=evaluator) for t in ("dev", "qa", "prod")}     # three separate workspaces

def run(args, target=None, cwd=BUNDLE, check=True, quiet=False):
    e = {**os.environ, "DATABRICKS_TOKEN": "local", "DATABRICKS_CLI": CLI, "PYTHONPATH": str(ROOT)}
    if target:
        e["DATABRICKS_HOST"] = WS[target].host
    p = subprocess.run([str(a) for a in args], cwd=cwd, env=e, capture_output=True, text=True)
    out = clean(p.stdout + "\n" + p.stderr).replace(str(LAB), "$LAB")
    if not quiet:
        shown = " ".join(str(a) for a in args).replace(CLI, "databricks").replace(sys.executable, "python").replace(str(LAB), "$LAB")
        print(f"$ {shown}" + (f"      # {target} workspace" if target else ""))
        if out:
            print(out)
        print()
    if check and p.returncode != 0:
        raise RuntimeError(f"exit code {p.returncode}")
    return p.returncode, p.stdout

def bundle(target, *args, **kw):   return run([CLI, "bundle", *args, "-t", target], target=target, **kw)
def script(name, *args, target=None, **kw): return run([sys.executable, f"scripts/{name}", *args], target=target, **kw)
def git(*args, quiet=False):
    out = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True, check=True).stdout.strip()
    if quiet:
        return out
    if out:
        print(out)

def space_id(target):
    out = run([CLI, "bundle", "summary", "-t", target, "-o", "json"], target=target, quiet=True)[1]
    return json.loads(out[out.find("{"):])["resources"]["genie_spaces"]["freshcart_assistant"]["id"]

def live(target):
    return json.loads(WS[target].spaces[space_id(target)]["serialized_space"])

git("init", "-q", "-b", "main"); git("config", "user.name", "Dana Developer"); git("config", "user.email", "dana@freshcart.example")
git("add", "-A"); git("commit", "-q", "-m", "Genie space bundle v1")
git("log", "--oneline")
for t, w in WS.items():
    print(f"{t:<5} workspace at {w.host}")

ef1701d Genie space bundle v1
dev   workspace at http://127.0.0.1:51451
qa    workspace at http://127.0.0.1:53527
prod  workspace at http://127.0.0.1:33729


### 6.2 · First deployment to dev
The CLI uploads the bundle, creates the Genie space and the gate job, and sets their permissions. A second
`bundle plan` shows nothing left to do.

In [9]:
bundle("dev", "deploy")
bundle("dev", "plan")
bundle("dev", "summary");

$ databricks bundle deploy -t dev      # dev workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev/files...
Created genie_spaces.freshcart_assistant
Created jobs.genie_quality_gate
Created genie_spaces.freshcart_assistant.permissions
Created jobs.genie_quality_gate.permissions
Files: 6 uploaded, 0 deleted
Resources: 4 created, 0 changed, 0 deleted, 0 unchanged

$ databricks bundle plan -t dev      # dev workspace
Plan: 0 to add, 0 to change, 0 to delete, 4 unchanged



$ databricks bundle summary -t dev      # dev workspace
Name: freshcart-genie
Target: dev
Workspace:
  User: dana@freshcart.example
  Path: /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev
Resources:
  Genie Spaces:
    freshcart_assistant:
      Name: FreshCart Sales & Stock Assistant [DEV]
      URL:  http://127.0.0.1:51451/genie/rooms/dfc9ae0bab634ed0886a722610bc4dbd?w=1234567890123456
  Jobs:
    genie_quality_gate:
      Name: FreshCart Genie quality gate [DEV]
      URL:  http://127.0.0.1:51451/jobs/1001?w=1234567890123456



### 6.3 · Dana improves the space in the dev UI; dev-sync captures it as a version
Dana adds a suggested question and sharpens an instruction in the Genie UI (`ui_edit` simulates that). The hourly
**genie-dev-sync** workflow runs `sync_from_workspace.py -t dev`, which uses `bundle summary` and
`bundle generate genie-space`, converts the catalog back to `${var.catalog}`, and writes the space file. The change
is committed on `genie/dev-sync` and **tagged**, so this state of dev can always be restored.

In [10]:
def add_line(content, text):
    if content and not content[-1].endswith("\n"):
        content[-1] += "\n"
    content.append(text)

def good_edit(s):
    s["config"]["sample_questions"].append({"id": "a" * 32, "question": ["Which stores opened in FY2025?"]})
    ti = s["instructions"]["text_instructions"][0]
    ti["content"] = [l.replace("Round percentages to one decimal place in the summary text.",
                               "Round percentages to one decimal place and money to whole dollars in the summary text.") for l in ti["content"]]

WS["dev"].ui_edit(space_id("dev"), good_edit)

def dev_sync(stamp):
    # what genie-dev-sync does on the genie/dev-sync branch: export dev; if it changed, commit + tag (+ PR)
    script("sync_from_workspace.py", "-t", "dev", target="dev")
    if not git("status", "--porcelain", "--", "genie_bundle/resources/freshcart_assistant.space.yml", quiet=True):
        print("dev matches git: nothing to record"); return None
    git("commit", "-q", "-am", f"Genie: changes made in the dev space (snapshot {stamp})")
    tag = f"genie-dev-snapshot-{stamp}"
    git("tag", tag); print("tagged", tag)
    return tag

git("checkout", "-q", "-B", "genie/dev-sync", "main")
snap1 = dev_sync("20261002T0900Z")

$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 2 change(s)
+ added sample question: Which stores opened in FY2025?
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- Say when a result covers comparable stores only.\n",
      -    "- Round percentages to one decimal place in the summary text.\n",
      +    "- Round percentages to one decimal place and money to whole dollars in the summary text.\n",
           "\n",

tagged genie-dev-snapshot-20261002T0900Z


### 6.4 · A second UI edit, with a mistake nobody notices
Dana edits the trusted example *"What were net sales and margin by region last week?"* and changes
`MEASURE(net_sales)` into `MEASURE(gross_sales)` by accident. Valid SQL, so nothing complains. The next dev-sync
records it as another version:

In [11]:
def the_mistake(s):
    for e in s["instructions"]["example_question_sqls"]:
        if "net sales and margin by region" in "".join(e["question"]):
            e["sql"] = [l.replace("MEASURE(net_sales)", "MEASURE(gross_sales)") for l in e["sql"]]

WS["dev"].ui_edit(space_id("dev"), the_mistake)
snap2 = dev_sync("20261002T1000Z")
print("\nversion history of the dev space:")
git("log", "--oneline", "--decorate", "genie/dev-sync")

$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 1 change(s)
~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(net_sales)        AS net_sales,\n",
      +    "       MEASURE(gross_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",

tagged genie-dev-snapshot-20261002T1000Z

version history of the dev space:
ee4e166 (HEAD -> genie/dev-sync, tag: genie-dev-snapshot-20261002T1000Z) Genie: changes made in the dev space (snapshot 20261002T1000Z)
e52a21a (tag: genie-dev-snapshot-20261002T0900Z) Genie: changes made in the dev space (snapshot 20261002T0900Z)
ef1701d (main) Genie space bundle v1


### 6.5 · The pull request
**genie-ci** runs on the sync pull request: the change list, the static checks, every SQL against the local data.
All pass: the SQL is valid and returns numbers. That is why SQL checks alone are not enough, and why QA has a
benchmark gate. The reviewer approves; the PR is merged.

In [12]:
script("validate_space.py", "--diff-against", "main", cwd=BUNDLE)
script("validate_space.py", "--sql")
git("checkout", "-q", "main"); git("merge", "-q", "--no-ff", "genie/dev-sync", "-m", "Merge pull request #21 from genie/dev-sync")
git("log", "--oneline", "--graph", "-5")

$ python scripts/validate_space.py --diff-against main
changes versus main (3):
+ added sample question: Which stores opened in FY2025?
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- Say when a result covers comparable stores only.\n",
      -    "- Round percentages to one decimal place in the summary text.\n",
      +    "- Round percentages to one decimal place and money to whole dollars in the summary text.\n",
           "\n",
~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(net_sales)        AS net_sales,\n",
      +    "       MEASURE(gross_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",



$ python scripts/validate_space.py --sql
space content 31af41e90086: {'table': 2, 'metric view': 2, 'sample question': 5, 'text instruction': 1, 'example SQL': 6, 'trusted function': 1, 'SQL filter': 2, 'SQL expression': 1, 'benchmark': 17}
OK

*   5bca3fb Merge pull request #21 from genie/dev-sync
|\  
| * ee4e166 Genie: changes made in the dev space (snapshot 20261002T1000Z)
| * e52a21a Genie: changes made in the dev space (snapshot 20261002T0900Z)
|/  
* ef1701d Genie space bundle v1


### 6.6 · The release: QA catches the mistake
Merging to `main` starts **genie-release**. The function below runs the same commands as the workflow's jobs. The
only stand-in: the stand-in workspaces can't execute jobs, so where the workflow runs
`databricks bundle run genie_quality_gate -t qa`, we run that job's task (`src/genie_quality_gate.py`) directly
against the qa workspace. It is the same code, using the Databricks SDK.

In [13]:
def gate(target, *extra):
    # the task of the bundle job genie_quality_gate, run directly against the stand-in workspace
    return run([sys.executable, "src/genie_quality_gate.py", "--space-id", space_id(target), "--min-graded", "4", *extra],
               target=target, check=False)[0] == 0

def release(approve=True, drift_flags=()):
    sha = git("rev-parse", "--short", "HEAD", quiet=True)
    print(f"########## genie-release for commit {sha} ##########\n")
    print("## dev-snapshot: capture UI edits before redeploying dev")
    git("checkout", "-q", "-B", "genie/dev-sync", "main"); dev_sync("release-" + sha); git("checkout", "-q", "main")
    print("## dev"); bundle("dev", "deploy")
    print("## qa")
    bundle("qa", "plan")
    plan = bundle("qa", "plan", "-o", "json", quiet=True)[1]; (LAB / "plan.qa.json").write_text(plan)
    if script("check_drift.py", LAB / "plan.qa.json", "-t", "qa", "--backup-dir", LAB / "backup", *drift_flags, check=False)[0]:
        print("STOP: drift gate"); return None
    bundle("qa", "deploy")
    print("## qa benchmark gate   (CI: databricks bundle run genie_quality_gate -t qa)")
    if not gate("qa"):
        print("STOP: the qa gate failed, prod is not touched"); return None
    print("## qa smoke test       (CI: databricks bundle run genie_quality_gate -t qa --params mode=smoke)")
    gate("qa", "--mode", "smoke")
    if not approve:
        print("WAIT: prod deployment needs approval"); return None
    print("## prod (approved by Priya, required reviewer of the prod environment)")
    bundle("prod", "plan")
    plan = bundle("prod", "plan", "-o", "json", quiet=True)[1]; (LAB / "plan.prod.json").write_text(plan)
    if script("check_drift.py", LAB / "plan.prod.json", "-t", "prod", "--backup-dir", LAB / "backup", *drift_flags, check=False)[0]:
        print("STOP: drift gate"); return None
    bundle("prod", "deploy")
    print("## prod smoke test     (CI: databricks bundle run genie_quality_gate -t prod --params mode=smoke)")
    gate("prod", "--mode", "smoke")
    tag = f"genie-prod-{len(git('tag', '-l', 'genie-prod-*', quiet=True).split()) + 1:03d}-{sha}"
    git("tag", tag); print("TAG", tag)
    return tag

release()

########## genie-release for commit 5bca3fb ##########

## dev-snapshot: capture UI edits before redeploying dev


$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 0 change(s)

dev matches git: nothing to record
## dev


$ databricks bundle deploy -t dev      # dev workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged

## qa
$ databricks bundle plan -t qa      # qa workspace
create genie_spaces.freshcart_assistant
create genie_spaces.freshcart_assistant.permissions
create jobs.genie_quality_gate
create jobs.genie_quality_gate.permissions

Plan: 4 to add, 0 to change, 0 to delete, 0 unchanged



$ python scripts/check_drift.py $LAB/plan.qa.json -t qa --backup-dir $LAB/backup
freshcart_assistant: plan action create



$ databricks bundle deploy -t qa      # qa workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/qa/files...
Created genie_spaces.freshcart_assistant
Created jobs.genie_quality_gate
Created genie_spaces.freshcart_assistant.permissions
Created jobs.genie_quality_gate.permissions
Files: 6 uploaded, 0 deleted
Resources: 4 created, 0 changed, 0 deleted, 0 unchanged

## qa benchmark gate   (CI: databricks bundle run genie_quality_gate -t qa)


$ python src/genie_quality_gate.py --space-id e9e4ef593f1b4707a4887ccd838b9c53 --min-graded 4      # qa workspace
benchmark run f39f5c0b62214768ab5349f363e12173: 3/4 graded correct, 1 bad, 13 need review
  BAD  What were net sales and margin by region last week?
       SINGLE_CELL_DIFFERENCE
GATE FAILED: 1 benchmark(s) answered incorrectly (allowed 0); accuracy 75% is below 95%

STOP: the qa gate failed, prod is not touched


The gate compared the answer to the trusted example question with the benchmark's correct SQL: the numbers differ
(gross vs net sales), the benchmark is `BAD` (`SINGLE_CELL_DIFFERENCE`), the job fails, and **prod is never touched**.

### 6.7 · Undo in dev: restore the snapshot from before the mistake
Dana restores the space file from the snapshot tag taken before the mistake and deploys it to dev with the bundle.
This is exactly what **genie-rollback** does (`environment = dev`, `to = genie-dev-snapshot-...`, `scope = space`).

In [14]:
git("checkout", "-q", "-b", "genie/fix-net-sales")
git("checkout", snap1, "--", "genie_bundle/resources/freshcart_assistant.space.yml")
bundle("dev", "deploy")
restored = T.neutralise(live("dev"), "freshcart_dev")
print("dev now equals snapshot", snap1, ":", T.content_hash(restored) == T.content_hash(T.load_space(SPACE_FILE)))
print("\nwhat the fix changes compared with main:")
script("validate_space.py", "--diff-against", "main")
git("commit", "-q", "-am", "Genie: restore net sales in the region example")
git("checkout", "-q", "main"); git("merge", "-q", "--no-ff", "genie/fix-net-sales", "-m", "Merge pull request #22 from genie/fix-net-sales")

$ databricks bundle deploy -t dev      # dev workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged



dev now equals snapshot genie-dev-snapshot-20261002T0900Z : True

what the fix changes compared with main:
$ python scripts/validate_space.py --diff-against main
changes versus main (1):
~ changed example SQL: What were net sales and margin by region last week?
           "SELECT region,\n",
      -    "       MEASURE(gross_sales)        AS net_sales,\n",
      +    "       MEASURE(net_sales)        AS net_sales,\n",
           "       MEASURE(gross_margin)     AS gross_margin,\n",



Now the release passes QA, is approved, reaches prod, passes the smoke test and is tagged:

In [15]:
release1 = release()

########## genie-release for commit b43881b ##########

## dev-snapshot: capture UI edits before redeploying dev


$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 0 change(s)

dev matches git: nothing to record
## dev


$ databricks bundle deploy -t dev      # dev workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev/files...
Files: 1 uploaded, 0 deleted
Resources: 0 created, 0 changed, 0 deleted, 4 unchanged

## qa


$ databricks bundle plan -t qa      # qa workspace
update genie_spaces.freshcart_assistant

Plan: 0 to add, 1 to change, 0 to delete, 3 unchanged



$ python scripts/check_drift.py $LAB/plan.qa.json -t qa --backup-dir $LAB/backup
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/qa.freshcart_assistant.live.space.yml (content 31af41e90086)



...

$ python src/genie_quality_gate.py --space-id e9e4ef593f1b4707a4887ccd838b9c53 --min-graded 4      # qa workspace
benchmark run bbcf5fbd4b78488ca4452ff447590e9b: 4/4 graded correct, 0 bad, 13 need review
GATE PASSED

## qa smoke test       (CI: databricks bundle run genie_quality_gate -t qa --params mode=smoke)


$ python src/genie_quality_gate.py --space-id e9e4ef593f1b4707a4887ccd838b9c53 --min-graded 4 --mode smoke      # qa workspace
smoke test: 'What were net sales and margin by region last week?' -> COMPLETED with SQL
  SELECT region,
         MEASURE(net_sales)        AS net_sales,
         MEASURE(gross_margin)     AS gross_margin,
         MEASURE(gross_margin_pct) AS gross_margin_pct
  FROM   freshcart_qa.semantic.sales_metrics
  WHERE  is_last_completed_fiscal_week
  GROUP BY ALL
  ORDER BY net_sales DESC
SMOKE TEST PASSED

## prod (approved by Priya, required reviewer of the prod environment)
$ databricks bundle plan -t prod      # prod workspace
create genie_spaces.freshcart_assistant
create genie_spaces.freshcart_assistant.permissions
create jobs.genie_quality_gate
create jobs.genie_quality_gate.permissions

Plan: 4 to add, 0 to change, 0 to delete, 0 unchanged



$ python scripts/check_drift.py $LAB/plan.prod.json -t prod --backup-dir $LAB/backup
freshcart_assistant: plan action create



$ databricks bundle deploy -t prod      # prod workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/prod/files...
Created genie_spaces.freshcart_assistant
Created jobs.genie_quality_gate
Created genie_spaces.freshcart_assistant.permissions
Created jobs.genie_quality_gate.permissions
Files: 6 uploaded, 0 deleted
Resources: 4 created, 0 changed, 0 deleted, 0 unchanged

## prod smoke test     (CI: databricks bundle run genie_quality_gate -t prod --params mode=smoke)


$ python src/genie_quality_gate.py --space-id 84742ec09c5e4c4da0bf4e5dabafed3b --min-graded 4 --mode smoke      # prod workspace
smoke test: 'What were net sales and margin by region last week?' -> COMPLETED with SQL
  SELECT region,
         MEASURE(net_sales)        AS net_sales,
         MEASURE(gross_margin)     AS gross_margin,
         MEASURE(gross_margin_pct) AS gross_margin_pct
  FROM   freshcart.semantic.sales_metrics
  WHERE  is_last_completed_fiscal_week
  GROUP BY ALL
  ORDER BY net_sales DESC
SMOKE TEST PASSED

TAG genie-prod-001-b43881b


### 6.8 · Someone edits prod directly in the UI: drift
An analyst adds an instruction straight into the prod space. A plain `bundle deploy` would silently overwrite it.
The release pipeline runs `bundle plan` first: the space's etag moved, so `check_drift.py` stops the deployment,
shows the edit and saves the live space as a backup.

In [16]:
WS["prod"].ui_edit(space_id("prod"), lambda s: add_line(s["instructions"]["text_instructions"][0]["content"], "Always mention the weather."))
bundle("prod", "plan")
plan = bundle("prod", "plan", "-o", "json", quiet=True)[1]; (LAB / "plan.prod.json").write_text(plan)
code_, _ = script("check_drift.py", LAB / "plan.prod.json", "-t", "prod", "--backup-dir", LAB / "backup", check=False)
print("exit code:", code_)

$ databricks bundle plan -t prod      # prod workspace
update genie_spaces.freshcart_assistant

Plan: 0 to add, 1 to change, 0 to delete, 3 unchanged



$ python scripts/check_drift.py $LAB/plan.prod.json -t prod --backup-dir $LAB/backup
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/prod.freshcart_assistant.live.space.yml (content 40c1c06c066e)
  DRIFT: edited outside the bundle since the last deploy (etag 1 -> 2)
  live space versus the definition in git:
    ~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
               "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
          -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
          +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
          +    "Always mention the weather."
             ],

Refusing to deploy over edits made in prod. 

| Option | When | How |
|---|---|---|
| **Adopt** the edit | it is a good change | `sync_from_workspace.py -t prod` on a branch → pull request → normal release |
| **Overwrite** it | it shouldn't be there | run genie-release with input **allow_drift** (the backup artifact keeps the edit) |

Here it shouldn't be there. Overwrite, with the backup kept:

In [17]:
script("check_drift.py", LAB / "plan.prod.json", "-t", "prod", "--backup-dir", LAB / "backup", "--allow-drift")
bundle("prod", "deploy")
backup = T.load_space(LAB / "backup" / "prod.freshcart_assistant.live.space.yml")
print("the analyst's edit is preserved in the backup:", any("weather" in l for l in backup["instructions"]["text_instructions"][0]["content"]))
print("prod is back to the release:", T.content_hash(T.neutralise(live("prod"), "freshcart")) == T.content_hash(T.load_space(SPACE_FILE)))

$ python scripts/check_drift.py $LAB/plan.prod.json -t prod --backup-dir $LAB/backup --allow-drift
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/prod.freshcart_assistant.live.space.yml (content 40c1c06c066e)
  DRIFT: edited outside the bundle since the last deploy (etag 1 -> 2)
  live space versus the definition in git:
    ~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
               "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
          -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
          +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
          +    "Always mention the weather."
             ],

--allow-drift: the deploy will

$ databricks bundle deploy -t prod      # prod workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/prod/files...
Updated genie_spaces.freshcart_assistant
Files: 0 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged

the analyst's edit is preserved in the backup: True


prod is back to the release: True


### 6.9 · An incident: a release the benchmarks couldn't catch
A change that keeps every SQL answer right but makes the summaries useless ("one sentence, no explanations") passes
all gates and reaches prod. Business users complain on Monday. **genie-rollback**: `environment = prod`,
`to = previous`, `scope = space`.

In [18]:
WS["dev"].ui_edit(space_id("dev"), lambda s: add_line(s["instructions"]["text_instructions"][0]["content"], "Answer in one sentence, no explanations."))
git("checkout", "-q", "-B", "genie/dev-sync", "main"); dev_sync("20261005T0800Z")
git("checkout", "-q", "main"); git("merge", "-q", "--no-ff", "genie/dev-sync", "-m", "Merge pull request #23 from genie/dev-sync")
release2 = release()

$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 1 change(s)
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
      -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
      +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
      +    "Answer in one sentence, no explanations."
         ],

tagged genie-dev-snapshot-20261005T0800Z
########## genie-release for commit f6bafbc ##########

## dev-snapshot: capture UI edits before redeploying dev


$ python scripts/sync_from_workspace.py -t dev      # dev workspace
synced dev into genie_bundle/resources/freshcart_assistant.space.yml: 0 change(s)

dev matches git: nothing to record
## dev


$ databricks bundle deploy -t dev      # dev workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/dev/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged

## qa
$ databricks bundle plan -t qa      # qa workspace
update genie_spaces.freshcart_assistant

Plan: 0 to add, 1 to change, 0 to delete, 3 unchanged



$ python scripts/check_drift.py $LAB/plan.qa.json -t qa --backup-dir $LAB/backup
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/qa.freshcart_assistant.live.space.yml (content 91ba5bcbf106)



$ databricks bundle deploy -t qa      # qa workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/qa/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged

## qa benchmark gate   (CI: databricks bundle run genie_quality_gate -t qa)


...

$ python src/genie_quality_gate.py --space-id e9e4ef593f1b4707a4887ccd838b9c53 --min-graded 4 --mode smoke      # qa workspace
smoke test: 'What were net sales and margin by region last week?' -> COMPLETED with SQL
  SELECT region,
         MEASURE(net_sales)        AS net_sales,
         MEASURE(gross_margin)     AS gross_margin,
         MEASURE(gross_margin_pct) AS gross_margin_pct
  FROM   freshcart_qa.semantic.sales_metrics
  WHERE  is_last_completed_fiscal_week
  GROUP BY ALL
  ORDER BY net_sales DESC
SMOKE TEST PASSED

## prod (approved by Priya, required reviewer of the prod environment)


$ databricks bundle plan -t prod      # prod workspace
update genie_spaces.freshcart_assistant

Plan: 0 to add, 1 to change, 0 to delete, 3 unchanged



$ python scripts/check_drift.py $LAB/plan.prod.json -t prod --backup-dir $LAB/backup
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/prod.freshcart_assistant.live.space.yml (content 91ba5bcbf106)



$ databricks bundle deploy -t prod      # prod workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/prod/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged

## prod smoke test     (CI: databricks bundle run genie_quality_gate -t prod --params mode=smoke)


$ python src/genie_quality_gate.py --space-id 84742ec09c5e4c4da0bf4e5dabafed3b --min-graded 4 --mode smoke      # prod workspace
smoke test: 'What were net sales and margin by region last week?' -> COMPLETED with SQL
  SELECT region,
         MEASURE(net_sales)        AS net_sales,
         MEASURE(gross_margin)     AS gross_margin,
         MEASURE(gross_margin_pct) AS gross_margin_pct
  FROM   freshcart.semantic.sales_metrics
  WHERE  is_last_completed_fiscal_week
  GROUP BY ALL
  ORDER BY net_sales DESC
SMOKE TEST PASSED

TAG genie-prod-002-f6bafbc


In [19]:
print("Monday 08:55, complaints. genie-rollback: prod, to=previous, scope=space\n")
tags = git("tag", "-l", "genie-prod-*", "--sort=-refname", quiet=True).split()      # same command as genie-rollback
previous = tags[1]
print("releases (newest first):", tags, "-> restoring", previous, "\n")
git("checkout", previous, "--", "genie_bundle/resources/freshcart_assistant.space.yml")
plan = bundle("prod", "plan", "-o", "json", quiet=True)[1]; (LAB / "plan.prod.json").write_text(plan)
script("check_drift.py", LAB / "plan.prod.json", "-t", "prod", "--backup-dir", LAB / "backup", "--allow-drift")
bundle("prod", "deploy")
gate("prod", "--mode", "smoke")
expected = T.load_space(SPACE_FILE)
print("prod content equals", previous, ":", T.content_hash(T.neutralise(live("prod"), "freshcart")) == T.content_hash(expected))
git("checkout", "HEAD", "--", "genie_bundle/resources/freshcart_assistant.space.yml")   # the workflow commits nothing

Monday 08:55, complaints. genie-rollback: prod, to=previous, scope=space

releases (newest first): ['genie-prod-002-f6bafbc', 'genie-prod-001-b43881b'] -> restoring genie-prod-001-b43881b 



$ python scripts/check_drift.py $LAB/plan.prod.json -t prod --backup-dir $LAB/backup --allow-drift
freshcart_assistant: plan action update
  backup of the live space: $LAB/backup/prod.freshcart_assistant.live.space.yml (content 9e6cb98124a4)



$ databricks bundle deploy -t prod      # prod workspace
Uploading bundle files to /Workspace/Users/dana@freshcart.example/.bundle/freshcart-genie/prod/files...
Updated genie_spaces.freshcart_assistant
Files: 1 uploaded, 0 deleted
Resources: 0 created, 1 changed, 0 deleted, 3 unchanged



$ python src/genie_quality_gate.py --space-id 84742ec09c5e4c4da0bf4e5dabafed3b --min-graded 4 --mode smoke      # prod workspace
smoke test: 'What were net sales and margin by region last week?' -> COMPLETED with SQL
  SELECT region,
         MEASURE(net_sales)        AS net_sales,
         MEASURE(gross_margin)     AS gross_margin,
         MEASURE(gross_margin_pct) AS gross_margin_pct
  FROM   freshcart.semantic.sales_metrics
  WHERE  is_last_completed_fiscal_week
  GROUP BY ALL
  ORDER BY net_sales DESC
SMOKE TEST PASSED



prod content equals genie-prod-001-b43881b : True


Prod is restored, with the **same space id**, so users keep their conversations. Then fix forward on `main`, for
example `git revert` of the bad merge, so the next release doesn't bring the change back:

In [20]:
git("revert", "--no-edit", "-m", "1", "HEAD", quiet=True)
git("log", "--oneline", "-3")

d3ab5c0 Revert "Merge pull request #23 from genie/dev-sync"
f6bafbc Merge pull request #23 from genie/dev-sync
9683e29 Genie: changes made in the dev space (snapshot 20261005T0800Z)


### 6.10 · The audit trail
Every state of dev, every reviewed change and every release is in git; each prod and qa deployment also has its
plan and backup as workflow artifacts, and GitHub records who approved it.

In [21]:
git("log", "--oneline", "--decorate", "--graph", "--all", "-20")
print()
print("dev snapshots:", git("tag", "-l", "genie-dev-snapshot-*", quiet=True).split())
print("prod releases:", git("tag", "-l", "genie-prod-*", quiet=True).split())
print("\nbackups taken before deploys:", sorted(p.name for p in (LAB / "backup").glob("*")))

* d3ab5c0 (HEAD -> main) Revert "Merge pull request #23 from genie/dev-sync"
*   f6bafbc (tag: genie-prod-002-f6bafbc, genie/dev-sync) Merge pull request #23 from genie/dev-sync
|\  
| * 9683e29 (tag: genie-dev-snapshot-20261005T0800Z) Genie: changes made in the dev space (snapshot 20261005T0800Z)
|/  
*   b43881b (tag: genie-prod-001-b43881b) Merge pull request #22 from genie/fix-net-sales
|\  
| * 63219a2 (genie/fix-net-sales) Genie: restore net sales in the region example
|/  
*   5bca3fb Merge pull request #21 from genie/dev-sync
|\  
| * ee4e166 (tag: genie-dev-snapshot-20261002T1000Z) Genie: changes made in the dev space (snapshot 20261002T1000Z)
| * e52a21a (tag: genie-dev-snapshot-20261002T0900Z) Genie: changes made in the dev space (snapshot 20261002T0900Z)
|/  
* ef1701d Genie space bundle v1

dev snapshots: ['genie-dev-snapshot-20261002T0900Z', 'genie-dev-snapshot-20261002T1000Z', 'genie-dev-snapshot-20261005T0800Z']
prod releases: ['genie-prod-001-b43881b', 'genie-prod-002-

In [22]:
print("What changed between the two prod releases:")
def at(ref):
    out = subprocess.run(["git", "-C", str(REPO), "show", f"{ref}:genie_bundle/resources/freshcart_assistant.space.yml"],
                         capture_output=True, text=True, check=True).stdout
    p = LAB / "tmp.yml"; p.write_text(out); return T.load_space(p)
print("\n".join(T.diff(at(release1), at(release2))))

What changed between the two prod releases:
~ changed text instruction: About FreshCart and this agent FreshCart is a grocery retail...
           "- What were net sales and margin by region last week? Last week always means the last completed fiscal week, Sunday to Saturday.\n",
      -    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales."
      +    "- How are we trading year to date versus last year? Total business comparison. For comparable-store growth use fn_like_for_like_sales.\n",
      +    "Answer in one sentence, no explanations."
         ],


In [23]:
for w in WS.values():
    w.stop()
shutil.rmtree(LAB, ignore_errors=True)
print("lab cleaned up")

lab cleaned up


---
# 7 · Going live: one-time setup

### 7.1 · Workspaces, catalogs and data
* Separate workspaces for dev, qa and prod are recommended (one workspace with separate catalogs also works: the
  targets only differ in `catalog` and in which host you deploy to).
* Each environment's catalog (`freshcart_dev`, `freshcart_qa`, `freshcart`) must contain the gold tables, the two
  metric views and the like-for-like function the space uses (`databricks/`). The data layer is promoted with its
  own bundle/targets in the same way; deploy it before the space.
* Pin each target's `workspace.host` in `databricks.yml` once known: then a qa credential can never deploy to prod.
* Set `warehouse_id`'s lookup name per target if your SQL warehouse isn't called "Serverless Starter Warehouse".

### 7.2 · Identities and permissions
| Identity | Permissions |
|---|---|
| One **service principal per environment** (member of `freshcart-genie-deployers`) | deploys the bundle (CAN_MANAGE on the space and job, via the bundle); `CAN_USE` on the warehouse; `USE CATALOG/SCHEMA`, `SELECT` on the data, `EXECUTE` on the function (benchmarks run as this identity) |
| `freshcart-genie-developers` | `CAN_MANAGE` in dev (edit in the UI), `CAN_VIEW` in qa/prod |
| `freshcart-business-users` | `CAN_RUN` on the space (set by the bundle); `SELECT` on the data (Genie queries as the user; row filters apply) |

Nobody but the service principals can change qa or prod, and if someone does, the drift gate catches it.

### 7.3 · GitHub → Databricks without secrets
1. For each service principal, add a **federation policy** trusting GitHub Actions: issuer
   `https://token.actions.githubusercontent.com`, subject `repo:<org>/<repo>:environment:<env>` (see Databricks'
   *Enable workload identity federation for GitHub Actions* for the audience value).
2. GitHub → Settings → Environments: **dev**, **qa**, **prod**, each with variables `DATABRICKS_HOST` and
   `DATABRICKS_CLIENT_ID`. On **prod**: *Required reviewers* and *Deployment branches: main*.
3. Branch protection on `main`: pull request + 1 approval + status checks **demo / run-demo** and **genie-ci / checks**.
4. The workflows set `DATABRICKS_AUTH_TYPE: github-oidc` and `permissions: id-token: write`; the CLI exchanges the
   GitHub token for a Databricks token. (Alternative: OAuth M2M with a `DATABRICKS_CLIENT_SECRET` environment secret.)

### 7.4 · First release
```bash
cd genie_bundle
databricks auth login --host https://<dev-workspace> --profile freshcart-dev
databricks bundle validate -t dev -p freshcart-dev
databricks bundle deploy   -t dev -p freshcart-dev            # creates "FreshCart Sales & Stock Assistant [DEV]"
python scripts/sync_from_workspace.py -t dev                  # round trip: commit what Databricks returns
git commit -am "Genie: adopt the space as Databricks exports it" && git push   # PR -> merge -> genie-release
```
The round trip matters: a few optional fields in the first definition (metric-view data sources, entity-matching
flags, synonyms on SQL expressions) follow Databricks solution guidance; the export from your workspace is
authoritative and becomes the reviewed baseline.

Already built a space by hand? Bind it to the bundle instead of creating a new one:
`databricks bundle deployment bind freshcart_assistant <space-id> -t dev`, then `sync_from_workspace.py -t dev`.

---
# 8 · The CI/CD workflows

| Workflow | Trigger | Steps (all `databricks bundle ...` + the scripts) |
|---|---|---|
| **demo** (`ci.yml`) | every push and pull request | the whole demo and the full test suite, once, including the bundle integration test (real CLI against the stand-in) |
| **genie-ci** | pull request touching the bundle | validate_space `--sql` · `bundle validate` for every target · change list |
| **genie-release** | merge to main (`genie_bundle/**`), or manual with `allow_drift` / `allow_destroy` | dev-snapshot → **dev** deploy → **qa** plan, drift gate, backup, deploy, `bundle run genie_quality_gate` (+ smoke) → approval → **prod** plan, drift gate, backup, deploy, smoke (auto-restore of the previous release on failure), tag |
| **genie-rollback** | manual: environment, `to` (previous / tag / commit), `scope` (space / bundle), reason | checkout version → plan → backup → deploy → smoke |
| **genie-dev-sync** | hourly, manual, and called by genie-release | sync_from_workspace -t dev → commit + `genie-dev-snapshot-*` tag → sync PR |

| Safety rule | Enforced by |
|---|---|
| Reviewed changes only | branch protection on `main` + demo and genie-ci required |
| Same content in every environment | every job deploys the same commit (`github.sha`) |
| Prod only after QA passed | `prod` job: `needs: qa`, `if: needs.qa.outputs.gated == 'true'` |
| Human approval for prod | `prod` GitHub environment, required reviewers |
| No silent overwrite of UI edits | `bundle plan -o json` → `check_drift.py` (exit 2) |
| No accidental deletion of the space (and its conversations) | `check_drift.py` (exit 3); and `bundle deploy` itself refuses destructive changes without `--auto-approve`, which the workflows never pass |
| One deployment at a time | `concurrency: genie-release` (release and rollback) |
| No stored secrets | GitHub OIDC → workload identity federation |

<details>
<summary><b>📄 genie-ci: <code>.github/workflows/genie-ci.yml</code></b> (click to open)</summary>

```yaml
# Genie space bundle: checks on every pull request that touches it (no Databricks workspace needed)
#
#  1. the space definition is valid, canonical and uses ${var.catalog}; every SQL in it runs
#  2. `databricks bundle validate` for every target
#  3. reviewers get a readable list of what changed in the space (job summary)
#
# The test suite (including the bundle integration test with the real Databricks CLI) runs once, in ci.yml.
# Make "genie-ci / checks" and "demo / run-demo" required status checks on main.
name: genie-ci

on:
  pull_request:
    paths:
      - "genie_bundle/**"
      - "semantic/**"
      - "tests/fake_workspace.py"
      - ".github/workflows/genie-*.yml"
  push:
    branches: [main]
    paths:
      - "genie_bundle/**"

permissions:
  contents: read

env:
  DATABRICKS_CLI_VERSION: "1.18.0"

jobs:
  checks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0                       # the base branch, for the change list

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - uses: databricks/setup-cli@main
        with:
          version: ${{ env.DATABRICKS_CLI_VERSION }}

      - run: pip install -r requirements.txt -r genie_bundle/requirements-dev.txt

      - name: Validate the space definition (+ every SQL on the local warehouse)
        run: python genie_bundle/scripts/validate_space.py --sql

      - name: bundle validate, every target (against the stand-in workspace)
        run: |
          python tests/fake_workspace.py 8765 & sleep 3
          cd genie_bundle
          for t in sandbox dev qa prod; do
            DATABRICKS_HOST=http://127.0.0.1:8765 DATABRICKS_TOKEN=local databricks bundle validate -t "$t"
          done

      - name: What changed in the Genie space
        if: github.event_name == 'pull_request'
        run: python genie_bundle/scripts/validate_space.py --diff-against "origin/${{ github.base_ref }}"
```
</details>

<details>
<summary><b>📄 genie-release: <code>.github/workflows/genie-release.yml</code></b> (click to open)</summary>

```yaml
# Genie space bundle: release  main -> dev -> qa (benchmark gate) -> prod (approval)
#
# Every environment is deployed with `databricks bundle deploy -t <target>` from the SAME commit.
# Before each deploy: `bundle plan -o json` -> check_drift.py (refuses to overwrite UI edits in qa/prod,
# refuses to delete or recreate the space, saves the live space as a backup artifact).
# Gates run as the bundle's own job: `databricks bundle run genie_quality_gate`.
#
# One-time setup (notebooks/Genie_CICD_with_Declarative_Automation_Bundles.ipynb, part 7):
#   GitHub environments dev, qa, prod (prod: required reviewers, deployment branch main), each with
#   variables DATABRICKS_HOST and DATABRICKS_CLIENT_ID (its deployment service principal).
#   Auth is GitHub OIDC -> Databricks workload identity federation: no secrets stored.
# Until DATABRICKS_HOST is set for an environment, its steps are skipped with a notice.
name: genie-release

on:
  push:
    branches: [main]
    paths:
      - "genie_bundle/**"
  workflow_dispatch:
    inputs:
      allow_drift:
        description: Overwrite edits made in the qa/prod UI since the last deploy (a backup is kept)
        type: boolean
        default: false
      allow_destroy:
        description: Allow a plan that deletes or recreates the Genie space (its conversations are lost)
        type: boolean
        default: false

concurrency:
  group: genie-release                         # shared with genie-rollback: one deployment at a time
  cancel-in-progress: false

permissions:
  contents: write                              # push release tags
  id-token: write                              # GitHub OIDC token for Databricks
  pull-requests: write                         # the dev snapshot may open a sync PR

env:
  DATABRICKS_CLI_VERSION: "1.18.0"
  DATABRICKS_AUTH_TYPE: github-oidc
  DRIFT_FLAGS: ${{ inputs.allow_drift && '--allow-drift' || '' }} ${{ inputs.allow_destroy && '--allow-destroy' || '' }}

jobs:
  # 1. Capture any UI edits in dev into git before dev is redeployed (nothing is ever lost).
  dev-snapshot:
    uses: ./.github/workflows/genie-dev-sync.yml
    secrets: inherit

  # 2. Bring dev in line with main.
  dev:
    needs: dev-snapshot
    runs-on: ubuntu-latest
    environment: dev
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
    defaults:
      run:
        working-directory: genie_bundle
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
        with:
          version: ${{ env.DATABRICKS_CLI_VERSION }}
      - id: cfg
        run: |
          if [ -z "$DATABRICKS_HOST" ]; then echo "::notice::dev is not connected (no DATABRICKS_HOST); skipping"; echo "ready=false" >> "$GITHUB_OUTPUT"; else echo "ready=true" >> "$GITHUB_OUTPUT"; fi
      - if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle validate -t dev
      - if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle deploy -t dev

  # 3. QA: plan, drift gate, deploy, benchmark gate, smoke test.
  qa:
    needs: dev
    runs-on: ubuntu-latest
    environment: qa
    outputs:
      gated: ${{ steps.gate.outcome == 'success' }}
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
    defaults:
      run:
        working-directory: genie_bundle
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - uses: databricks/setup-cli@main
        with:
          version: ${{ env.DATABRICKS_CLI_VERSION }}
      - run: pip install -r requirements-dev.txt
      - id: cfg
        run: |
          if [ -z "$DATABRICKS_HOST" ]; then echo "::notice::qa is not connected (no DATABRICKS_HOST); skipping"; echo "ready=false" >> "$GITHUB_OUTPUT"; else echo "ready=true" >> "$GITHUB_OUTPUT"; fi
      - if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle validate -t qa
      - name: Plan, drift gate and backup of the live space
        if: steps.cfg.outputs.ready == 'true'
        run: |
          databricks bundle plan -t qa
          databricks bundle plan -t qa -o json > plan.qa.json
          python scripts/check_drift.py plan.qa.json -t qa --backup-dir backup $DRIFT_FLAGS
      - if: steps.cfg.outputs.ready == 'true'
        uses: actions/upload-artifact@v4
        with:
          name: genie-qa-pre-deploy-${{ github.run_id }}
          path: |
            genie_bundle/plan.qa.json
            genie_bundle/backup/
          retention-days: 90
      - name: Deploy
        if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle deploy -t qa
      - id: gate
        name: Benchmark gate (bundle job)
        if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle run genie_quality_gate -t qa
      - name: Smoke test (bundle job)
        if: steps.cfg.outputs.ready == 'true'
        run: databricks bundle run genie_quality_gate -t qa --params mode=smoke

  # 4. Prod: same commit, after approval (required reviewers on the prod environment).
  prod:
    needs: qa
    if: needs.qa.outputs.gated == 'true'       # never without a passed qa gate
    runs-on: ubuntu-latest
    environment: prod
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
    defaults:
      run:
        working-directory: genie_bundle
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0                       # release tags, for the automatic rollback
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - uses: databricks/setup-cli@main
        with:
          version: ${{ env.DATABRICKS_CLI_VERSION }}
      - run: pip install -r requirements-dev.txt
      - run: databricks bundle validate -t prod
      - name: Plan, drift gate and backup of the live space
        run: |
          databricks bundle plan -t prod
          databricks bundle plan -t prod -o json > plan.prod.json
          python scripts/check_drift.py plan.prod.json -t prod --backup-dir backup $DRIFT_FLAGS
      - uses: actions/upload-artifact@v4
        with:
          name: genie-prod-pre-deploy-${{ github.run_id }}
          path: |
            genie_bundle/plan.prod.json
            genie_bundle/backup/
          retention-days: 400
      - name: Deploy
        run: databricks bundle deploy -t prod
      - name: Smoke test (rolls the space back to the previous release if it fails)
        run: |
          if ! databricks bundle run genie_quality_gate -t prod --params mode=smoke; then
            PREVIOUS=$(git tag -l 'genie-prod-*' --sort=-refname | head -n 1)
            echo "::error::smoke test failed on prod; restoring the space from ${PREVIOUS:-nothing}"
            if [ -n "$PREVIOUS" ]; then
              git checkout "$PREVIOUS" -- resources/freshcart_assistant.space.yml
              databricks bundle deploy -t prod
            fi
            exit 1
          fi
      - name: Tag the release
        run: |
          TAG="genie-prod-$(date -u +%Y%m%d.%H%M)-${GITHUB_SHA::7}"
          git tag "$TAG" "$GITHUB_SHA"
          git push origin "$TAG"
          echo "Released $TAG" >> "$GITHUB_STEP_SUMMARY"
```
</details>

<details>
<summary><b>📄 genie-rollback: <code>.github/workflows/genie-rollback.yml</code></b> (click to open)</summary>

```yaml
# Genie space bundle: rollback (run manually from the Actions tab)
#
#   to    = previous                       the release before the latest genie-prod-* tag
#         = <git tag or commit>            e.g. genie-prod-20261001.0915-1a2b3c4, or genie-dev-snapshot-<time>
#   scope = space   (default)  restore only resources/freshcart_assistant.space.yml from that version and
#                              deploy it with today's bundle (the job, permissions, ... stay current)
#         = bundle             deploy the whole genie_bundle/ folder as it was at that version
#
# It is a normal `databricks bundle deploy`, approved like any prod deploy, and the live space is backed
# up first. Afterwards, fix forward on main (for example `git revert`) so the next release doesn't
# bring the bad version back.
name: genie-rollback

on:
  workflow_dispatch:
    inputs:
      environment:
        description: Environment to roll back
        type: choice
        options: [prod, qa, dev]
        required: true
      to:
        description: "previous | <git tag or commit>"
        default: previous
        required: true
      scope:
        description: What to restore
        type: choice
        options: [space, bundle]
        default: space
      reason:
        description: Why (kept in the run log)
        required: true

concurrency:
  group: genie-release
  cancel-in-progress: false

permissions:
  contents: read
  id-token: write

jobs:
  rollback:
    runs-on: ubuntu-latest
    environment: ${{ inputs.environment }}
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      TARGET: ${{ inputs.environment }}        # inputs reach the shell as variables, never as code
      ROLLBACK_TO: ${{ inputs.to }}
      SCOPE: ${{ inputs.scope }}
      REASON: ${{ inputs.reason }}
    defaults:
      run:
        working-directory: genie_bundle
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - uses: databricks/setup-cli@main
        with:
          version: "1.18.0"
      - run: pip install -r requirements-dev.txt
      - name: Resolve the version to restore
        run: |
          echo "Rollback of $TARGET to '$ROLLBACK_TO' (scope $SCOPE) requested by $GITHUB_ACTOR: $REASON"
          if [ "$ROLLBACK_TO" = "previous" ]; then
            REF=$(git tag -l 'genie-prod-*' --sort=-refname | sed -n 2p)
            [ -n "$REF" ] || { echo "::error::there is no earlier genie-prod-* release"; exit 1; }
          else
            REF="$ROLLBACK_TO"
          fi
          git rev-parse --verify "$REF^{commit}" >/dev/null
          echo "REF=$REF" >> "$GITHUB_ENV"
          echo "Restoring $TARGET to $REF" >> "$GITHUB_STEP_SUMMARY"
      - name: Check out that version
        run: |
          if [ "$SCOPE" = "space" ]; then
            git checkout "$REF" -- resources/freshcart_assistant.space.yml
          else
            git checkout "$REF" -- .
          fi
          git status --short
      - name: Plan and backup of the live space
        run: |
          databricks bundle plan -t "$TARGET"
          databricks bundle plan -t "$TARGET" -o json > plan.json
          python scripts/check_drift.py plan.json -t "$TARGET" --backup-dir backup --allow-drift
      - uses: actions/upload-artifact@v4
        with:
          name: genie-${{ inputs.environment }}-pre-rollback-${{ github.run_id }}
          path: |
            genie_bundle/plan.json
            genie_bundle/backup/
          retention-days: 400
      - name: Deploy the restored version
        run: databricks bundle deploy -t "$TARGET"
      - name: Smoke test
        if: inputs.environment != 'dev'
        run: databricks bundle run genie_quality_gate -t "$TARGET" --params mode=smoke
```
</details>

<details>
<summary><b>📄 genie-dev-sync: <code>.github/workflows/genie-dev-sync.yml</code></b> (click to open)</summary>

```yaml
# Genie space bundle: version every change made in the dev Genie UI
#
# Hourly, on demand, and at the start of every release:
#   1. scripts/sync_from_workspace.py -t dev     (databricks bundle summary + bundle generate genie-space)
#      exports the dev space and writes resources/freshcart_assistant.space.yml with ${var.catalog}
#   2. if it changed: commit it on the branch genie/dev-sync, tag the commit
#      genie-dev-snapshot-<UTC time> (the version history of dev: every snapshot stays reachable and can
#      be restored with genie-rollback, environment dev, to = that tag), and open/update a pull request.
#
# Note: pull requests created with the default GITHUB_TOKEN don't trigger other workflows. To run
# genie-ci on the sync PR automatically, store a GitHub App token or fine-grained PAT as GENIE_SYNC_TOKEN.
name: genie-dev-sync

on:
  schedule:
    - cron: "17 * * * *"
  workflow_dispatch:
  workflow_call:

concurrency:
  group: genie-dev-sync
  cancel-in-progress: false

permissions:
  contents: write
  pull-requests: write
  id-token: write

jobs:
  sync:
    runs-on: ubuntu-latest
    environment: dev
    env:
      DATABRICKS_HOST: ${{ vars.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ vars.DATABRICKS_CLIENT_ID }}
      DATABRICKS_AUTH_TYPE: github-oidc
      GH_TOKEN: ${{ secrets.GENIE_SYNC_TOKEN || github.token }}
    steps:
      - uses: actions/checkout@v4
        with:
          ref: main
          token: ${{ secrets.GENIE_SYNC_TOKEN || github.token }}
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - uses: databricks/setup-cli@main
        with:
          version: "1.18.0"
      - run: pip install -r genie_bundle/requirements-dev.txt
      - id: cfg
        run: |
          if [ -z "$DATABRICKS_HOST" ]; then echo "::notice::dev is not connected (no DATABRICKS_HOST); nothing to sync"; echo "ready=false" >> "$GITHUB_OUTPUT"; else echo "ready=true" >> "$GITHUB_OUTPUT"; fi
      - name: Export the dev space into the bundle
        if: steps.cfg.outputs.ready == 'true'
        run: python genie_bundle/scripts/sync_from_workspace.py -t dev
      - name: Commit, tag and open the sync pull request
        if: steps.cfg.outputs.ready == 'true'
        run: |
          FILE=genie_bundle/resources/freshcart_assistant.space.yml
          if git diff --quiet -- "$FILE"; then echo "dev matches main"; exit 0; fi
          python genie_bundle/scripts/validate_space.py || echo "::warning::the dev space has validation errors; see the PR checks"
          STAMP=$(date -u +%Y%m%dT%H%MZ)
          git config user.name "genie-dev-sync"
          git config user.email "genie-dev-sync@users.noreply.github.com"
          git checkout -B genie/dev-sync
          git add "$FILE"
          git commit -m "Genie: changes made in the dev space (snapshot $STAMP)"
          git tag "genie-dev-snapshot-$STAMP"
          git push --force origin genie/dev-sync
          git push origin "genie-dev-snapshot-$STAMP"
          gh pr view genie/dev-sync >/dev/null 2>&1 || gh pr create --base main --head genie/dev-sync \
            --title "Genie: sync changes made in dev" \
            --body "Changes made in the dev Genie space UI, exported by genie-dev-sync. Review the change list in the checks, then merge to release to qa and prod."
          echo "Snapshot genie-dev-snapshot-$STAMP" >> "$GITHUB_STEP_SUMMARY"
```
</details>

---
# 9 · Runbook

**Make a change.** Edit the dev space in the UI → wait for the hourly sync PR (or run genie-dev-sync) → review the
change list → merge → genie-release → approve prod. Editing the YAML directly instead? Deploy your branch to your
`sandbox` (or `dev`) target first, so dev and git agree.

**Add a benchmark for every bug.** Add the question and its correct SQL as a benchmark in dev; from then on every
release proves the bug stays fixed (and the nightly prod run watches it on live data).

**Emergency rollback.** Actions → genie-rollback → `prod`, `previous`, `space`, reason → approve. Then fix forward on
main (`git revert`), or the next release brings the change back.

**Undo in dev.** genie-rollback → `dev`, `to = genie-dev-snapshot-<time>`.

**"Refusing to deploy over edits made in prod" (exit 2).** Someone edited prod in the UI. Adopt (sync from prod on
a branch, PR) or overwrite (rerun genie-release with allow_drift; the backup artifact keeps the edit).

**"Refusing: the plan would delete/recreate" (exit 3).** A change would replace the space and lose every
conversation (renaming the resource key, for example). Use `databricks bundle deployment bind` to move the existing
space to the new key instead, or rerun with allow_destroy if you really mean it.

**"QA gate failed".** The job run lists each `BAD` benchmark and the reason. Fix in dev (often: restore the last good
snapshot), PR, release.

**What the bundle does not version.** Conversations and feedback (they stay in the space, which is updated in place,
never recreated), the warehouse itself, and the data (versioned by its own pipeline).

---
# 10 · Summary

* **One bundle, four targets.** The Genie space, its permissions and its quality-gate job are a Declarative
  Automation Bundle; `databricks bundle deploy -t <target>` is the only way anything reaches a workspace.
* **One definition for every environment.** The space content is a bundle variable that uses `${var.catalog}`.
* **Every change has a version.** Dev UI edits become tagged commits (bundle generate + sync); releases are tagged;
  every qa/prod deploy keeps a backup of what it replaced.
* **Promotion is the same commit moving forward**, with a benchmark gate in QA, approval for prod and a smoke test.
* **Rollback is a normal bundle deploy** of an earlier version, through the same safety checks.

| Term | Meaning |
|---|---|
| Declarative Automation Bundle (DAB) | YAML definition of Databricks resources, deployed per target with the Databricks CLI |
| Target | one environment of a bundle (sandbox, dev, qa, prod) with its own variables, mode and permissions |
| `genie_spaces` resource | the bundle resource type for a Genie space (direct engine, CLI 1.14+) |
| Serialized space | the JSON content of a Genie space (instructions, examples, benchmarks...) |
| Complex variable | a bundle variable holding a whole structure; here, the space content |
| `bundle generate genie-space` | export an existing space from a workspace into bundle files |
| `bundle plan -o json` | what a deploy would do, including the live (remote) state |
| Drift | the live space changed outside the bundle since the last deploy (etag moved) |
| Benchmark gate | Genie benchmark evaluation with pass rules, run as a bundle job |